In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# =====================================================================
# 1. GESTION DES CHEMINS (COLAB / LOCAL)
# =====================================================================
if 'google.colab' in sys.modules:
    if not Path('/content/Stat_app').exists():
        !git clone https://github.com/Aymanyah/Stat_app.git
    PROJECT_DIR = Path('/content/Stat_app')
else:
    PROJECT_DIR = Path.cwd().parent 

DATA_RAW = PROJECT_DIR / "data" / "raw" / "all_leagues_merged_transformed.csv"
DATA_CLEAN_DIR = PROJECT_DIR / "data" / "clean"
DATA_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================================
# 2. CHARGEMENT ET CORRECTION DES ENCODAGES
# =====================================================================
df = pd.read_csv(DATA_RAW, encoding="utf8")

# Nettoyage des noms de colonnes (Espaces cachés et encodage)
df.columns = [col.encode('latin1').decode('utf8', errors='ignore') for col in df.columns]
df.columns = df.columns.str.strip()

# Fonction de correction du texte (Noms avec des accents bizarres)
def fix_encoding(x):
    if isinstance(x, str):
        try:
            return x.encode("latin1").decode("utf8")
        except:
            return x
    return x

for col in ["player", "team", "league", "nation", "pos"]:
    if col in df.columns:
        df[col] = df[col].apply(fix_encoding)
        df[col] = df[col].str.strip()

df_clean = df.drop_duplicates()

# =====================================================================
# 3. STRATÉGIE D'AGRÉGATION (Joueurs transférés en cours d'année)
# =====================================================================
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["season"]]

def smart_agg(col):
    # Les ratios et pourcentages sont moyennés
    if "percent" in col.lower() or "%" in col or "ratio" in col.lower():
        return "mean"
    # Les statistiques de volume sont sommées
    if col in ["Minutes de jeu", "G", "G-PK", "A", "Tirs", "Tirs cadrés"]:
        return "sum"
    return "mean"

agg_dict = {col: smart_agg(col) for col in numeric_cols}

# Pour le club et la ligue, on garde la dernière équipe de la saison
categorical_cols = [c for c in ["team", "league", "nation", "pos"] if c in df_clean.columns]
for col in categorical_cols:
    agg_dict[col] = "last"

# =====================================================================
# 4. CRÉATION DU FORMAT LONG (Le Tidy Data)
# =====================================================================
# On regroupe par joueur ET par saison. Chaque ligne = 1 joueur pour 1 saison.
df_final = df_clean.groupby(["player", "season"]).agg(agg_dict).reset_index()

# =====================================================================
# 5. RETRAITEMENTS FINAUX ET SAUVEGARDE
# =====================================================================
# Mettre en format 0-1 les valeurs en pourcentages (si elles sont en 0-100)
percent_cols = [c for c in df_final.columns if "%" in c]
for col in percent_cols:
    if df_final[col].mean(skipna=True) > 1:
        df_final[col] = df_final[col] / 100

fichier_sauvegarde = DATA_CLEAN_DIR / "dataset_aggregated.csv"
df_final.to_csv(fichier_sauvegarde, index=False)

print(f"✅ Nettoyage terminé ! {df_final.shape[0]} lignes générées.")
print(f"📁 Données au format LONG sauvegardées dans : {fichier_sauvegarde}")

✅ Nettoyage terminé ! 6530 lignes générées.
📁 Données au format LONG sauvegardées dans : /home/onyxia/Stat_app/data/clean/dataset_aggregated.csv


In [2]:
import pandas as pd

df_final = pd.read_csv('/home/onyxia/Stat_app/data/clean/all_leagues_merged_final_df.csv')
df_trans = pd.read_csv('/home/onyxia/Stat_app/data/raw/all_leagues_merged_transformed.csv')

print(f"Final : {df_final.shape}")
print(f"Transformed : {df_trans.shape}")

# Voir les colonnes qui diffèrent
diff_cols = set(df_final.columns) ^ set(df_trans.columns)
print(f"Colonnes différentes : {diff_cols}")

Final : (10015, 106)
Transformed : (10015, 44)
Colonnes différentes : {'Expected xG', 'Tirs cadrÃ©s', 'Touches Live', 'Carries PrgDist', 'xG / Tir', '% passes moyennes rÃ©ussies', 'Per 90 Minutes xG+xAG', 'Blocks Sh', 'Carries Carries', 'Performance G-PK', 'G+A', 'Passes reÃ§ues', 'G-PK', 'Distance totale des passes > 10m avant effectuÃ©es', 'Tackles Def 3rd', 'Short Cmp', 'Dribbles subis perdus', 'Tackles Tkl', 'Total PrgDist', 'Medium Cmp%', 'Touches Att Pen', 'Carries TotDist', 'Passes avant >10m', 'Long Cmp', '% duels aÃ©riens gagnÃ©s', 'Playing Time 90s', 'Per 90 Minutes G+A', 'Expected xA', 'Performance Gls', 'Tackles Att 3rd', 'Challenges Tkl', 'Per 90 Minutes npxG', 'Total Cmp%', ' npxG+xAG', '% dribble  rÃ©ussis', 'Ast', 'Tacles rÃ©ussis + interceptions', 'ChevauchÃ©e avant  >5m', 'Carries CPA', ' xG', 'Take-Ons Att', 'Medium Cmp', 'Total Cmp', 'Carries PrgC', 'Short Att', 'Err', 'Expected npxG', 'PrgP', 'Progression PrgP', 'Playing Time Min', 'G', 'Blocks Blocks', 'PPA', 'Tac